In [36]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import (StratifiedKFold, RepeatedStratifiedKFold,
                                     cross_val_score, cross_val_predict, GridSearchCV)
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.ensemble import VotingClassifier

train_df = pd.read_csv('data/train.csv', index_col=[0])
test_df = pd.read_csv('data/test.csv', index_col=[0])

## treino

In [37]:
train_df.head()

,pelvic_incidence,pelvic_tilt,lumbar_lordosis_angle,sacral_slope,pelvic_radius,degree_spondylolisthesis,class
0,39.056951,10.060991,25.015378,28.995960,114.405425,4.564259,Hernia
1,68.832021,22.218482,50.092194,46.613539,105.985135,-3.530317,Hernia
2,49.712859,9.652075,28.317406,40.060784,108.168725,7.918501,Hernia
3,40.250200,13.921907,25.124950,26.328293,130.327871,2.230652,Hernia
4,53.432928,15.864336,37.165934,37.568592,120.567523,5.988551,Hernia


In [38]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 217 entries, 0 to 216
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   pelvic_incidence          217 non-null    float64
 1   pelvic_tilt               217 non-null    float64
 2   lumbar_lordosis_angle     217 non-null    float64
 3   sacral_slope              217 non-null    float64
 4   pelvic_radius             217 non-null    float64
 5   degree_spondylolisthesis  217 non-null    float64
 6   class                     217 non-null    str    
dtypes: float64(6), str(1)
memory usage: 12.0 KB


In [39]:
train_df.describe()

,pelvic_incidence,pelvic_tilt,lumbar_lordosis_angle,sacral_slope,pelvic_radius,degree_spondylolisthesis
count,217.000000,217.000000,217.000000,217.000000,217.000000,217.000000
mean,60.458343,17.541355,52.660081,42.916988,118.073520,27.715448
std,18.045677,10.786425,19.226267,13.631236,13.920744,40.965137
min,26.147921,-6.554948,14.000000,13.516568,70.082575,-11.058179
25%,46.236399,9.976664,36.329137,33.366366,110.660701,1.517203
50%,57.145851,16.417462,51.000000,42.447102,118.151531,13.683047
75%,74.469082,22.432950,63.232302,51.766175,126.398188,49.159426
max,129.834041,49.431864,125.742385,121.429566,163.071041,418.543082


contagem de classes

In [40]:
train_df.iloc[:,-1].value_counts()

class
Spondylolisthesis    105
Normal                70
Hernia                42
Name: count, dtype: int64

ta desbalanceado. fato que precisa de estratificação

"A Kernel Density Estimation (KDE) plot is a type of plot that visualizes the estimated probability density function of a continuous variable. Unlike a histogram, which displays data using discrete bars grouped into intervals, a KDE plot represents the distribution as a smooth, continuous curve based on all data points." https://codefinity.com/courses/v2/61928667-6699-4af9-a146-b70e73fce3c5/d6354309-3168-44c6-b630-26cf849bd32f/f2dcae95-a3e5-4751-8730-870f0050101b

Eu tenho claramente variaveis numericas e continuas (pode asusmir inf valores dentro de um intervalo)

In [41]:
train_df = train_df[train_df['degree_spondylolisthesis'] < 400]

X são as features (6 atributos) e y eh a classe que sera prevista

In [42]:
X = train_df.drop(columns=['class'])
y = train_df['class']

In [43]:
X

,pelvic_incidence,pelvic_tilt,lumbar_lordosis_angle,sacral_slope,pelvic_radius,degree_spondylolisthesis
0,39.056951,10.060991,25.015378,28.995960,114.405425,4.564259
1,68.832021,22.218482,50.092194,46.613539,105.985135,-3.530317
2,49.712859,9.652075,28.317406,40.060784,108.168725,7.918501
3,40.250200,13.921907,25.124950,26.328293,130.327871,2.230652
4,53.432928,15.864336,37.165934,37.568592,120.567523,5.988551
...,...,...,...,...,...,...
212,54.600316,21.488974,29.360216,33.111342,118.343321,-1.471067
213,34.382299,2.062683,32.390820,32.319617,128.300199,-3.365516
214,45.075450,12.306951,44.583177,32.768499,147.894637,-8.941709
215,53.936748,20.721496,29.220534,33.215251,114.365845,-0.421010


In [44]:
y

0      Hernia
1      Hernia
2      Hernia
3      Hernia
4      Hernia
        ...  
212    Normal
213    Normal
214    Normal
215    Normal
216    Normal
Name: class, Length: 216, dtype: str

In [45]:
# mantend a msm ordem d colunas do treino 
X_test = test_df[X.columns]

In [46]:
cross_val = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [47]:
dummy = DummyClassifier(strategy="most_frequent")
score = cross_val_score(dummy, X, y, cv=cross_val, scoring="accuracy")

In [48]:
score.mean()

np.float64(0.48150105708245244)

In [49]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

cross_val = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "decision tree": make_pipeline(RobustScaler(), DecisionTreeClassifier(max_depth=40, random_state=42)),
    "knn(euclidian)": make_pipeline(RobustScaler(), KNeighborsClassifier(n_neighbors=10, metric='euclidean', weights='distance')),
}

In [50]:
results = {}
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=cross_val, scoring='accuracy', n_jobs=-1)
    results[name] = scores
    print(f"name={name:32s}, mean={scores.mean():.4f}, std={scores.std():.4f}")

name=decision tree                   , mean=0.8010, std=0.0430
name=knn(euclidian)                  , mean=0.8101, std=0.0727


/home/caio/miniconda3/envs/npb_analysis/lib/python3.14/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/home/caio/miniconda3/envs/npb_analysis/lib/python3.14/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/home/caio/miniconda3/envs/npb_analysis/lib/python3.14/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. R

# Decision Tree

In [51]:
params_grid = {
    'decisiontreeclassifier__max_depth': [2, 3, 4, 5, 6, 8, 10, 11, 12, 13, 14, 15],
    'decisiontreeclassifier__min_samples_leaf': [1, 3, 5, 10],
    'decisiontreeclassifier__ccp_alpha': [0.0, 0.01, 0.02, 0.05],
}

In [52]:
for scaler in [RobustScaler(), StandardScaler(), MinMaxScaler()]:
    pipe = make_pipeline(scaler, DecisionTreeClassifier(random_state=42))
    gs = GridSearchCV(estimator=pipe, param_grid=params_grid, cv=cross_val, scoring='accuracy', n_jobs=-1)
    gs.fit(X, y)

    print("best params:", gs.best_params_)
    print('best acc:', gs.best_score_)


/home/caio/miniconda3/envs/npb_analysis/lib/python3.14/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/home/caio/miniconda3/envs/npb_analysis/lib/python3.14/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/home/caio/miniconda3/envs/npb_analysis/lib/python3.14/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. R

best params: {'decisiontreeclassifier__ccp_alpha': 0.0, 'decisiontreeclassifier__max_depth': 2, 'decisiontreeclassifier__min_samples_leaf': 1}
best acc: 0.851797040169133
best params: {'decisiontreeclassifier__ccp_alpha': 0.0, 'decisiontreeclassifier__max_depth': 2, 'decisiontreeclassifier__min_samples_leaf': 1}
best acc: 0.851797040169133
best params: {'decisiontreeclassifier__ccp_alpha': 0.0, 'decisiontreeclassifier__max_depth': 2, 'decisiontreeclassifier__min_samples_leaf': 1}
best acc: 0.851797040169133


## KNN

In [53]:
knn_results = []
print("KNN")
for n in [2, 3, 4, 5, 10, 20, 30, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50]:
    for metric in ['minkowski', 'euclidean', 'manhattan', 'chebyshev']:
        for scaler in [RobustScaler(), StandardScaler()]:
            pipe = make_pipeline(RobustScaler(), KNeighborsClassifier(n_neighbors=n, metric=metric, weights='distance'))
            score = cross_val_score(pipe, X, y, cv=cross_val, scoring="accuracy", n_jobs=-1)
            print(f"knn: n={str(n):>9}, metric={metric}, score_mean={score.mean():.4f}, score_std={score.std():.4f}")
            knn_results.append({
                "n": n,
                "scaler": scaler,
                "metric": metric,
                "score_mean": score.mean(),
                "score_std": score.std(),
            })

KNN
knn: n=        2, metric=minkowski, score_mean=0.8238, score_std=0.0507
knn: n=        2, metric=minkowski, score_mean=0.8238, score_std=0.0507
knn: n=        2, metric=euclidean, score_mean=0.8238, score_std=0.0507
knn: n=        2, metric=euclidean, score_mean=0.8238, score_std=0.0507
knn: n=        2, metric=manhattan, score_mean=0.8377, score_std=0.0535
knn: n=        2, metric=manhattan, score_mean=0.8377, score_std=0.0535
knn: n=        2, metric=chebyshev, score_mean=0.8099, score_std=0.0319
knn: n=        2, metric=chebyshev, score_mean=0.8099, score_std=0.0319
knn: n=        3, metric=minkowski, score_mean=0.8007, score_std=0.0748
knn: n=        3, metric=minkowski, score_mean=0.8007, score_std=0.0748
knn: n=        3, metric=euclidean, score_mean=0.8007, score_std=0.0748
knn: n=        3, metric=euclidean, score_mean=0.8007, score_std=0.0748
knn: n=        3, metric=manhattan, score_mean=0.8145, score_std=0.0740
knn: n=        3, metric=manhattan, score_mean=0.8145, score

In [54]:
best_knn = max(knn_results, key=lambda c: (c['score_mean'], -c['score_std']))
best_knn

{'n': 4,
 'scaler': RobustScaler(),
 'metric': 'manhattan',
 'score_mean': np.float64(0.8424947145877377),
 'score_std': np.float64(0.07429718135647177)}

In [55]:
# final = DecisionTreeClassifier(max_depth=3, random_state=42)
# final.fit(X, y)
final = make_pipeline(RobustScaler(), KNeighborsClassifier(n_neighbors=best_knn['n'], metric=best_knn['metric'], weights='distance'))
final.fit(X, y)
preds = final.predict(X_test)
preds

array(['Spondylolisthesis', 'Spondylolisthesis', 'Hernia', 'Hernia',
       'Normal', 'Spondylolisthesis', 'Normal', 'Spondylolisthesis',
       'Hernia', 'Hernia', 'Hernia', 'Normal', 'Spondylolisthesis',
       'Normal', 'Spondylolisthesis', 'Spondylolisthesis', 'Normal',
       'Hernia', 'Normal', 'Spondylolisthesis', 'Normal',
       'Spondylolisthesis', 'Normal', 'Hernia', 'Spondylolisthesis',
       'Normal', 'Normal', 'Spondylolisthesis', 'Normal', 'Hernia',
       'Spondylolisthesis', 'Hernia', 'Normal', 'Spondylolisthesis',
       'Normal', 'Normal', 'Normal', 'Normal', 'Hernia',
       'Spondylolisthesis', 'Spondylolisthesis', 'Normal', 'Hernia',
       'Normal', 'Spondylolisthesis', 'Spondylolisthesis', 'Hernia',
       'Spondylolisthesis', 'Spondylolisthesis', 'Spondylolisthesis',
       'Spondylolisthesis', 'Normal', 'Spondylolisthesis',
       'Spondylolisthesis', 'Normal', 'Hernia', 'Normal',
       'Spondylolisthesis', 'Hernia', 'Hernia', 'Spondylolisthesis',
       'Sp

### Voting

In [56]:
simple_cv  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
repeated_cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=20, random_state=42)

In [57]:
def avaliar(nome, modelo, cv=repeated_cv, verbose=True):
    s = cross_val_score(modelo, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
    if verbose:
        print(f'{nome:44s} {s.mean():.4f}  (EP={s.std()/np.sqrt(len(s)):.4f})')
    return s.mean()

In [58]:
avaliar('Baseline (classe majoritária)', DummyClassifier(strategy='most_frequent'));

Baseline (classe majoritária)                0.4815  (EP=0.0009)


In [ ]:
from sklearn.preprocessing import PowerTransformer
nb_simples = GaussianNB()
#knn_power = make_pipeline(PowerTransformer(), KNeighborsClassifier(n_neighbors=5))
knn_power = make_pipeline(PowerTransformer(), KNeighborsClassifier(n_neighbors=best_knn['n'], metric=best_knn['metric'], weights='distance'))
#knn_otimizado = make_pipeline(RobustScaler(), KNeighborsClassifier(n_neighbors=best_knn['n'], metric=best_knn['metric'], weights='distance'))
dt_otimizada = DecisionTreeClassifier(max_depth=2, min_samples_leaf=1,ccp_alpha=0.0, random_state=42)
#best params: {'decisiontreeclassifier__ccp_alpha': 0.0, 'decisiontreeclassifier__max_depth': 2, 'decisiontreeclassifier__min_samples_leaf': 1}

final = VotingClassifier(
    estimators=[
        #('knn_power_2', knn_power_2),
        ('knn_power', knn_power),
        #('knn', knn_otimizado), 
        ('nb', nb_simples), 
        ('dt', dt_otimizada)
    ], 
    voting='soft'
)

tabela = {}
for nome, m in {'knn power': knn_power, 'NB Default': nb_simples, 'DT Default': dt_otimizada, 'VOTAÇÃO FINAL': final}.items():
    tabela[nome] = [cross_val_score(m, X, y, n_jobs=-1,
                    cv=RepeatedStratifiedKFold(n_splits=5, n_repeats=20, random_state=s)).mean()
                    for s in [7, 42, 59, 89, 101, 999, 10000]]

print("Acurácia média ao longo de 7 sementes de CV repetida (100 Folds cada):")
print(pd.DataFrame(tabela, index=['seed=7', 'seed=42', 'seed=59', 'seed=89', 'seed=101', 'seed=999', 'seed=10000']).T.round(4).to_string())

Acurácia média ao longo de 7 sementes de CV repetida (100 Folds cada):
               seed=7  seed=42  seed=59  seed=89  seed=101  seed=999  seed=10000
knn power      0.8278   0.8301   0.8299   0.8213    0.8280    0.8238      0.8268
NB Default     0.8467   0.8435   0.8467   0.8486    0.8472    0.8449      0.8478
DT Default     0.8158   0.8205   0.8134   0.8240    0.8227    0.8254      0.8273
VOTAÇÃO FINAL  0.8486   0.8511   0.8497   0.8500    0.8479    0.8514      0.8495


In [60]:
print(final)

VotingClassifier(estimators=[('knn_power',
                              Pipeline(steps=[('powertransformer',
                                               PowerTransformer()),
                                              ('kneighborsclassifier',
                                               KNeighborsClassifier(metric='manhattan',
                                                                    n_neighbors=4,
                                                                    weights='distance'))])),
                             ('nb', GaussianNB()),
                             ('dt',
                              DecisionTreeClassifier(class_weight='balanced',
                                                     max_depth=2,
                                                     random_state=42))],
                 voting='soft')


In [61]:
final.fit(X, y)
predicoes = final.predict(X_test)

In [62]:
def create_submission_file(predictions, test_df, submission_file_name="submission.csv"):
    submission_df = pd.DataFrame({"id": test_df.index, "Target": predictions})
    submission_df.to_csv(submission_file_name, index=False)
    print(f"Submission file '{submission_file_name}' created successfully.")


In [63]:
create_submission_file(predicoes, test_df)

Submission file 'submission.csv' created successfully.


In [64]:
#create_submission_file(preds, test_df, submission_file_name="submission_knn.csv")